# Gold - ecommerce_enderecos

Notebook para criação de tabelas Gold em Delta Lake e réplica opcional para SQL Server/Azure, mantendo padrão de consumo analítico via Looker.

Este notebook assume que as tabelas Silver e `squad1.dq_monitoring_logs` já foram criadas em Delta.

In [0]:
%run ../utils/utils

## Inicialização

In [0]:



import pyspark.sql.functions as F
from pyspark.sql.window import Window

print("Iniciando processamento da Camada Gold - Data Quality")

## VARIÁVEL DE CONTROLE

In [0]:
# ---------------------------------------------------------
# VARIÁVEL DE CONTROLE: Troque o nome da tabela aqui!
# ---------------------------------------------------------
TABELA_ALVO = "ecommerce_enderecos"

# Nomes dinâmicos das tabelas Gold geradas
tabela_gold_regras = f"gold_dq_regras_{TABELA_ALVO}"
tabela_gold_saude = f"gold_dq_saude_{TABELA_ALVO}"

print(f"===== INICIANDO PROCESSAMENTO DA CAMADA GOLD PARA: {TABELA_ALVO} =====")

## PROCESSAMENTO DAS REGRAS

In [0]:
# ==============================================================================
# 2. PROCESSAMENTO DE SAÚDE POR TABELA (Volume de registros entregues por Hora)
# ==============================================================================
print(f"\nProcessando Saúde por Hora para {TABELA_ALVO}...")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    
    df_gold_saude = df_silver \
        .withColumn("data_hora", F.date_trunc("hour", F.col("silver_processed_at"))) \
        .groupBy("data_hora") \
        .agg(
            # Como a Silver só armazena dados que passaram na validação,
            # o total de linhas É o total de registros limpos.
            F.count("*").alias("qtd_limpos") 
        ) \
        .withColumn("tabela", F.lit(TABELA_ALVO)) \
        .withColumn("qtd_total", F.col("qtd_limpos")) \
        .withColumn("perc_limpos", F.lit(100.0)) \
        .select("data_hora", "tabela", "qtd_total", "qtd_limpos", "perc_limpos") \
        .orderBy(F.col("data_hora").desc())
        
    print(f"-> Resumo de Saúde gerado com sucesso ({df_gold_saude.count()} horas processadas).")
else:
    print(f"-> Aviso: Tabela Silver {TABELA_ALVO} não encontrada.")
    df_gold_saude = None

##  PROCESSAMENTO DE SAÚDE POR TABELA

In [0]:
# ==============================================================================
# 1. PROCESSAMENTO DAS REGRAS (Dashboard 1: Falhas por Dia e Regra)
# ==============================================================================
print(f"Processando Resumo por Regra para {TABELA_ALVO}...")

if delta_existe(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS):
    # Lê os logs gerais e filtra EXCLUSIVAMENTE a tabela alvo
    df_logs = ler_delta(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)
    
    if df_logs.count() > 0:
        df_gold_regras = df_logs \
            .withColumn("data_execucao", F.to_date("timestamp_execucao")) \
            .groupBy("data_execucao", "tabela", "regra", "severidade") \
            .agg(
                F.sum("qtd_registros_falhos").alias("total_falhas"),
                F.sum("qtd_registros_total").alias("total_processado") 
            ) \
            .withColumn(
                "perc_falha", 
                F.round((F.col("total_falhas") / F.col("total_processado")) * 100, 2)
            ) \
            .orderBy(F.col("data_execucao").desc(), F.col("perc_falha").desc())
            
        print(f"-> Resumo por Regra gerado com sucesso ({df_gold_regras.count()} linhas).")
    else:
        print(f"-> Aviso: Nenhum log de falha encontrado para {TABELA_ALVO}.")
        df_gold_regras = None
else:
    print("-> Aviso: Tabela dq_monitoring_logs não encontrada no Data Lake.")
    df_gold_regras = None

## SALVAMENTO 1: DELTA LAKE

In [0]:
# ==============================================================================
# 3. SALVAMENTO 1: DELTA LAKE (A Fonte da Verdade na Camada Gold)
# ==============================================================================
print("\nIniciando salvamento no DELTA LAKE...")

if df_gold_regras is not None:
    gravar_delta(
        df=df_gold_regras,
        camada="gold",
        tabela=tabela_gold_regras,
        storage_opts=STORAGE_OPTIONS,
        mode="overwrite", 
        particionar=False
    )
    print(f"-> Sucesso: {tabela_gold_regras} gravada em delta (pasta gold).")

if df_gold_saude is not None:
    gravar_delta(
        df=df_gold_saude,
        camada="gold",
        tabela=tabela_gold_saude,
        storage_opts=STORAGE_OPTIONS,
        mode="overwrite",
        particionar=False
    )
    print(f"-> Sucesso: {tabela_gold_saude} gravada em delta (pasta gold).")

## SALVAMENTO 2: AZURE SQL SERVER

In [0]:
# ==============================================================================
# 4. SALVAMENTO 2: AZURE SQL SERVER (Camada de Serviço para o Looker)
# ==============================================================================
print("\nIniciando espelhamento no AZURE SQL SERVER...")

def salvar_sql_server(df, nome_tabela):
    try:
        # Passamos a porta padrão 1433 diretamente na configuração
        df.write \
            .format("sqlserver") \
            .option("host", JDBC_HOSTNAME) \
            .option("port", "1433") \
            .option("database", JDBC_DATABASE) \
            .option("dbtable", f"squad1.{nome_tabela}") \
            .option("user", JDBC_USERNAME) \
            .option("password", JDBC_PASSWORD) \
            .option("trustServerCertificate", "true") \
            .mode("overwrite") \
            .save()
            
        print(f"-> Sucesso: Tabela squad1.{nome_tabela} atualizada no SQL Server.")
    except Exception as e:
        print(f"-> Erro ao enviar {nome_tabela} ao SQL Server: {e}")

if df_gold_regras is not None:
    salvar_sql_server(df_gold_regras, tabela_gold_regras)

if df_gold_saude is not None:
    salvar_sql_server(df_gold_saude, tabela_gold_saude)

print(f"\n===== CAMADA GOLD FINALIZADA PARA {TABELA_ALVO} =====")